In [1]:
import pandas as pd
import numpy as np

In [2]:
demographic = pd.read_csv('cleaned dataset/api_data_aadhar_demographic.csv')

In [3]:
col_kids = 'age between 5 and 17'          # The Mandatory Update Group
col_adults = 'age 17 and above'   # The Proxy for "Active Households"

In [4]:
# We sum up all activity over time to get the "Total Compliance Status" of the district
compliance_df = demographic.groupby('district')[[col_kids, col_adults]].sum().reset_index()

In [5]:
# Metric: "For every 100 Adult Updates, how many Child Updates happened?"
# Formula: (Child Updates / Adult Updates) * 100
compliance_df['Compliance_Index'] = (compliance_df[col_kids] / compliance_df[col_adults]) * 100

# Handle infinite values (if a district has 0 adult updates)
compliance_df.replace([np.inf, -np.inf], 0, inplace=True)

display(compliance_df.head())

,district,age between 5 and 17,age 17 and above,Compliance_Index
0,5Th Cross,0,1,0.000000
1,Adilabad,9009,66292,13.589875
2,Agar Malwa,965,8919,10.819599
3,Agra,18477,157476,11.733216
4,Ahilyanagar,200,2218,9.017133


In [6]:
# Lower Index = Higher Risk (Parents updating, Kids NOT updating)
compliance_df['State_Rank'] = compliance_df['Compliance_Index'].rank(ascending=True)

display(compliance_df.sort_values('State_Rank').head(10))

,district,age between 5 and 17,age 17 and above,Compliance_Index,State_Rank
0,5Th Cross,0,1,0.0,15.5
797,Tiswadi,0,4,0.0,15.5
796,Tiruvarur,0,2,0.0,15.5
347,Kadiri Road,0,2,0.0,15.5
53,Balianta,0,1,0.0,15.5
56,Bally Jagachha,0,1,0.0,15.5
64,Bandipur,0,1,0.0,15.5
741,South Dumdum,0,8,0.0,15.5
736,South Twenty Four Parganas,0,5,0.0,15.5
107,Bicholim,0,4,0.0,15.5


In [7]:
# We define the Bottom 20% of districts as "CRITICAL Risk"
quantile_20 = compliance_df['Compliance_Index'].quantile(0.20)
quantile_50 = compliance_df['Compliance_Index'].quantile(0.50)

In [8]:
def assign_risk_category(val):
    if val < quantile_20:
        return 'CRITICAL (High Risk of Service Denial)'
    elif val < quantile_50:
        return 'Warning (Low Compliance)'
    else:
        return 'Safe (High Compliance)'

compliance_df['Risk_Category'] = compliance_df['Compliance_Index'].apply(assign_risk_category)

display(compliance_df.head())


,district,age between 5 and 17,age 17 and above,Compliance_Index,State_Rank,Risk_Category
0,5Th Cross,0,1,0.000000,15.5,CRITICAL (High Risk of Service Denial)
1,Adilabad,9009,66292,13.589875,557.0,Safe (High Compliance)
2,Agar Malwa,965,8919,10.819599,407.0,Warning (Low Compliance)
3,Agra,18477,157476,11.733216,464.0,Safe (High Compliance)
4,Ahilyanagar,200,2218,9.017133,299.0,Warning (Low Compliance)


In [9]:

compliance_df.sort_values('Compliance_Index', ascending=True, inplace=True)
compliance_df.to_csv('csv for power bi/biometric_compliance_risk.csv', index=False)


How to Visualize this in Power BI
You want to build a "Risk Leaderboard" to identify the worst-performing districts.

1. The "Compliance Gap" Chart (Bar Chart)
Visual Type: Clustered Bar Chart.

Y-Axis (Categories): district.

X-Axis (Values): Compliance_Index.

Legend (Color): Risk_Category.

Sort Order: Sort by Compliance_Index in Ascending order.

Result: The bars at the top will be the shortest (or red), representing the districts with the lowest child compliance. These are your "Service Denial" hotspots.

2. The "Gap" Matrix (Scatter Plot)
Visual Type: Scatter Chart.

X-Axis: age 17 and above (Volume of Adult Activity).

Y-Axis: age 5 to 17 (Volume of Child Updates).

Legend: Risk_Category.

Result:

Safe Districts: Follow a diagonal line (More Adults = More Kids).

CRITICAL Districts: Will hug the bottom-right corner (High Adult volume, but Flat/Zero Kid volume). This visually proves the gap.